In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\ADMIN\Documents\Guvi_Final_Project\RideFlow_Project/rideflow_datasets.csv")
df.head()

,ride_id,timestamp,pickup_zone,drop_zone,pickup_lat,pickup_long,drop_lat,drop_long,driver_id,customer_id,...,surge_multiplier,driver_rating,customer_rating,estimated_eta_min,actual_eta_min,ride_status,traffic_level,weather,driver_active,feedback_text
0,95.247911,2025-01-02 01:30:00,Anna Nagar,Adyar,12.880239,80.148410,13.028939,80.163941,1842.701958,6072.494896,...,1.001779,4.350624,4.037232,11.778023,18.304775,cancelled,low,clear,-0.036560,Driver was polite
1,439.187632,2025-01-05 12:45:00,T Nagar,Tambaram,13.092441,80.165458,13.142711,80.149376,1186.296422,5942.228896,...,1.193147,4.524196,3.324278,4.430894,13.343961,completed,low,cloudy,0.988999,Driver cancelled suddenly
2,876.685389,2025-01-09 23:00:00,Anna Nagar,Tambaram,12.817965,80.161839,12.943527,80.166040,1297.199801,5829.181415,...,2.008478,4.054085,4.979153,19.202891,12.039878,completed,low,rain,0.005750,Driver cancelled suddenly
3,275.337197,2025-01-03 19:30:00,T Nagar,Velachery,13.125103,80.143306,13.209127,80.126008,1765.474261,5429.619496,...,1.218528,3.689937,3.099466,18.711931,7.535792,completed,low,clear,1.023604,Good experience
4,106.743950,2025-01-02 02:30:00,Tambaram,Tambaram,13.143513,80.302596,13.078330,80.189672,1565.653849,5079.081677,...,1.497370,3.545512,3.073704,10.786351,12.104096,completed,high,cloudy,1.016716,Vehicle was not clean


# Data Preparation

In [2]:
# Date & Hour Columns
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour

In [3]:
# Encode pickup_zone and drop_zone
from sklearn.preprocessing import LabelEncoder

# Encode weather
le_pickup = LabelEncoder()
le_drop = LabelEncoder()
le_weather = LabelEncoder()
le_traffic = LabelEncoder()
df['pickup_zone_enc'] = le_pickup.fit_transform(df['pickup_zone'])
df['drop_zone_enc'] = le_drop.fit_transform(df['drop_zone'])
df['traffic_level_enc'] = le_traffic.fit_transform(df['traffic_level'])
df['weather_enc'] = le_weather.fit_transform(df['weather'])


from math import radians, sin, cos, sqrt, atan2
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2*np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

df['ride_distance_km'] = haversine(df['pickup_lat'], df['pickup_long'], df['drop_lat'], df['drop_long'])
df['eta_delay'] = df['actual_eta_min'] - df['estimated_eta_min']

# AI-Based Cancellation Risk Prediction

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

features_cancel = [
    'estimated_eta_min',
    'driver_rating',
    'surge_multiplier',
    'hour'
]

df['is_cancelled'] = df['ride_status'].apply(
    lambda x: 1 if x == "cancelled" else 0
)

X = df[features_cancel]
y = df['is_cancelled']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cancel_model = RandomForestClassifier()
cancel_model.fit(X_train, y_train)

df['cancellation_prob'] = cancel_model.predict_proba(X)[:,1]

# AI-Based ETA Prediction System

In [6]:
from sklearn.ensemble import RandomForestRegressor

features_eta = [
    'pickup_zone_enc',
    'drop_zone_enc',
    'traffic_level_enc',
    'ride_distance_km',
    'hour'
]

X_eta = df[features_eta]
y_eta = df['actual_eta_min']

eta_model = RandomForestRegressor()
eta_model.fit(X_eta, y_eta)

df['predicted_eta'] = eta_model.predict(X_eta)

# AI-Based Ride Matching Score System

In [7]:
# Building matching score
def calculate_match_score(row):
    return (
        (1 / (row['predicted_eta'] + 1)) * 0.4 +
        (1 - row['cancellation_prob']) * 0.4 +
        (row['driver_rating'] / 5) * 0.2
    )

df['match_score'] = df.apply(calculate_match_score, axis=1)

# Intelligent Driver Recommendation System

In [8]:
# Selecting best driver for zone
zone_drivers = df[df['pickup_zone'] == "Adyar"]

best_driver = zone_drivers.sort_values(
    by="match_score",
    ascending=False
).iloc[0]

print("Recommended Driver:", best_driver['driver_id'])

print(f"""
Reason:
Low ETA: {best_driver['predicted_eta']:.1f} mins
Low Cancellation Risk: {best_driver['cancellation_prob']:.2f}
High Rating: {best_driver['driver_rating']:.1f}
""")

Recommended Driver: 1193.7110862536158

Reason:
Low ETA: 2.7 mins
Low Cancellation Risk: 0.00
High Rating: 5.0



In [9]:
import joblib

# ===============================
# SAVE MODELS
# ===============================
joblib.dump(eta_model, "eta_model.pkl")
joblib.dump(cancel_model, "cancel_model.pkl")

# ===============================
# SAVE FEATURE
# ===============================
joblib.dump(features_eta, "eta_features.pkl")
joblib.dump(features_cancel, "cancel_features.pkl")

# ===============================
# SAVE ENCODERS
# ===============================
joblib.dump(le_pickup, "le_pickup.pkl")
joblib.dump(le_drop, "le_drop.pkl")
joblib.dump(le_weather, "le_weather.pkl")
joblib.dump(le_traffic, "le_traffic.pkl")

print("All models, features, and encoders saved successfully")

All models, features, and encoders saved successfully
